# Leakage Ablation -- Training (6 runs)

Retrains Model A (species) and Model B (freshness) on the **image-level** split, 3 seeds each. Every hyperparameter matches the main experiment; the split file is the only difference, which is what makes the comparison attributable to leakage.

Model C and the D variants are not retrained -- the two single-task models size the effect on both tasks.

Writes only under `08_Leakage_Ablation/`. `05_Checkpoints/` and `06_Results/` hold the main results and are not touched.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'

# Ablation artifacts live in their own Drive subtree, away from the main results.
ABLATION_DIR = '/content/drive/MyDrive/fish-freshness-mtl/leakage_ablation'
CHECKPOINT_DIR = f'{ABLATION_DIR}/checkpoints'
RESULTS_DIR = f'{ABLATION_DIR}/results'

import os
for d in ['/content/data', CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
!unzip -q -n "$DATASET_ZIP" -d /content/data

In [ ]:
import sys
sys.path.append('/content/repo/04_Src')

import pandas as pd
from train import train_single_task, TrainConfig

SPLIT_PATH = '/content/repo/08_Leakage_Ablation/02_manifests/split_manifest_imagelevel.csv'
split_df = pd.read_csv(SPLIT_PATH)

train_df = split_df[split_df.subset == 'train'].reset_index(drop=True)
val_df = split_df[split_df.subset == 'val'].reset_index(drop=True)
test_df = split_df[split_df.subset == 'test'].reset_index(drop=True)
len(train_df), len(val_df), len(test_df)

Confirm this really is the leaky split before spending GPU time -- a zero here would mean the wrong file was loaded.

In [ ]:
spanning = split_df.groupby('time_group')['subset'].nunique()
n_spanning = int((spanning > 1).sum())
assert n_spanning > 0, 'no group spans subsets -- this is not the image-level split'

train_groups = set(train_df.time_group)
contaminated = test_df[test_df.time_group.isin(train_groups)]
print(f'{n_spanning} time groups span subsets')
print(f'{len(contaminated)} test images ({len(contaminated)/len(test_df)*100:.1f}%) have a twin in train')

In [ ]:
import json

SEEDS = [42, 43, 44]
EXPERIMENTS = [
    {'name': 'ModelA_species', 'task': 'species'},
    {'name': 'ModelB_freshness', 'task': 'freshness'},
]

cfg = TrainConfig()  # identical to the main experiment, unchanged

for exp in EXPERIMENTS:
    for seed in SEEDS:
        run_name = f"imglevel_{exp['name']}_seed{seed}"
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'{run_name}.pt')
        if os.path.exists(ckpt_path):
            print(f'[{run_name}] checkpoint exists, skipping')
            continue
        print(f'[{run_name}] starting...')
        result = train_single_task(
            exp['task'], train_df=train_df, val_df=val_df, dataset_root=DATASET_ROOT,
            checkpoint_path=ckpt_path, seed=seed, cfg=cfg, run_name=run_name,
        )
        with open(os.path.join(RESULTS_DIR, f'{run_name}_history.json'), 'w') as f:
            json.dump(result['history'], f)

In [ ]:
print(len([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pt')]), 'checkpoints (expect 6)')